# City of Boston Public Notices - RAG Agent
Reads the `chroma_db/` which built by `notice_data_extraction.ipynb`. (Read only)

Connect to ChromaDB

In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# should match notice_data_extraction.ipynb
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_DB_PATH = "chroma_db"
COLLECTION_NAME = "public_notices"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DB_PATH,
)

c:\Users\shaik\Documents\Foundations of Gen AI\Project\CS6180_GenAI_Final_Project_City_of_Boston_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5822.20it/s]


Checking whats in the collection

In [8]:
print("chunks in collection:", vectorstore._collection.count())

sample = vectorstore.get(limit=3, include=["documents", "metadatas"])
for doc, meta in zip(sample["documents"], sample["metadatas"]):
    print("\n\n\n\n\n")
    print("TEXT:", doc[:200])
    print("SOURCE TYPE:", meta.get("source_type"), "| FILE:", meta.get("file_label"))
    print("NOTICE:", meta.get("notice_id"), "| URL:", meta.get("detail_url"))

chunks in collection: 12






TEXT: ORDER FOR A HEARING REGARDING THE FUTURE OF THE BOSTON HUMAN RIGHTS COMMISSION
WHEREAS, The Boston Human Rights Commission was established to ensure full and equal access to public services and accomm
SOURCE TYPE: pdf | FILE: Docket #1237
NOTICE: 16600326 | URL: https://www.boston.gov/public-notices/16600326






TEXT: ORDER FOR A HEARING REGARDING THE FUTURE OF THE BOSTON HUMAN RIGHTS COMMISSION
WHEREAS, The City of Boston's Equity Cabinet has undertaken significant work to advance equity, inclusion, language acces
SOURCE TYPE: pdf | FILE: Docket #1237
NOTICE: 16600326 | URL: https://www.boston.gov/public-notices/16600326






TEXT: ORDER FOR A HEARING REGARDING THE FUTURE OF THE BOSTON HUMAN RIGHTS COMMISSION
ORDERED: That  the  appropriate  committee  of  the  Boston  City  Council  hold  a  hearing regarding  the  future  of  
SOURCE TYPE: pdf | FILE: Docket #1237
NOTICE: 16600326 | URL: https://www.boston.gov/public-notices/16600326


Retrieval test (lower score is better)

In [9]:
for q in ["when is the hearing and where is it being held?",
          "who sponsored this order?",
          "how do I testify at the hearing?",
          "what does the human rights commission do?"]:
    print("\n#####", q)
    for doc, score in vectorstore.similarity_search_with_score(q, k=2):
        print(f"  [{score:.3f}] {doc.metadata.get('source_type')}: {doc.page_content[:140]}")


##### when is the hearing and where is it being held?
  [0.866] page_text: Members of the public are cordially invited to attend and testify in person or virtually. Those wishing to testify in person should arrive f
  [0.926] pdf: COMMITTEE HEARING NOTICE
July 15, 2026
The Boston City Council's Committee on Civil Rights, Racial Equity, and Immigrant Advancement will ho

##### who sponsored this order?
  [1.576] pdf: COMMITTEE HEARING NOTICE
rneghan.kavanaghboston.gov I Staff Telephone:
(617) 635-3040
Broadcast Live on Xfinity 8/RCN 82/ Fios 964 and strea
  [1.605] page_text: Order for a hearing regarding the future of the Boston Human Rights Commission. 
This matter was sponsored by Councilors Miniard Culpepper, 

##### how do I testify at the hearing?
  [0.573] page_text: Members of the public are cordially invited to attend and testify in person or virtually. Those wishing to testify in person should arrive f
  [0.805] pdf: COMMITTEE HEARING NOTICE
Public Testimony Members of the pu

Metadata filtering: same scores, smaller pool

In [10]:
query = "who sponsored this order?"

# Two separate searches, one per source_type. Scores are unchanged from the unfiltered run, only the set of eligible chunks differs.
for source_type in ["page_text", "pdf"]:
    print(f"\n### {source_type}")
    for doc, score in vectorstore.similarity_search_with_score(
        query, k=2, filter={"source_type": source_type}
    ):
        print(f"  [{score:.3f}] {doc.page_content[:130]}")


### page_text
  [1.605] Order for a hearing regarding the future of the Boston Human Rights Commission. 
This matter was sponsored by Councilors Miniard C
  [1.693] be made a part of the record and available to all Councilors. The public may watch this hearing via live stream at www.boston.gov/

### pdf
  [1.576] COMMITTEE HEARING NOTICE
rneghan.kavanaghboston.gov I Staff Telephone:
(617) 635-3040
Broadcast Live on Xfinity 8/RCN 82/ Fios 964
  [1.679] COMMITTEE HEARING NOTICE
Public Testimony Members of the public are cordially invited to attend and testify in person or virtually


OpenAI client

In [11]:
import os
import json
from openai import OpenAI

if not key.startswith("sk-"):
    raise ValueError(f"Key doesn't look valid: {len(key)} chars starting {key[:6]!r}")

os.environ["OPENAI_API_KEY"] = key
client = OpenAI()

LLM_MODEL = "gpt-4o-mini"

Query planning: split the question into search text + metadata filters

In [12]:
# fields that actually exist in our metadata.
ALLOWED_FILTERS = {"source_type", "cancelled", "notice_id"}

EXTRACT_PROMPT = """You convert a user's question about Boston public notices into a search plan.

Available metadata fields for filtering:
- source_type: "pdf" or "page_text"
- cancelled: true or false
- notice_id: integer

Return ONLY valid JSON, no markdown fences, in this shape:
{{"search_text": "<the topical part of the question to search semantically>",
  "filters": {{"<field>": <value>}}}}

Use "filters" only for constraints that map to the fields listed above.
Remove from search_text any wording that you converted into a filter.
Anything about dates, times, or neighborhoods should stay in search_text for now.
If there are no applicable filters, use an empty object.

Question: {question}"""


def extract_search_plan(question):
    """Returns (search_text, filters)."""
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": EXTRACT_PROMPT.format(question=question)}],
        response_format={"type": "json_object"},   
    )
    plan = json.loads(resp.choices[0].message.content)

    # drop filters on fields we dont store
    raw_filters = plan.get("filters") or {}
    filters = {k: v for k, v in raw_filters.items() if k in ALLOWED_FILTERS}
    dropped = set(raw_filters) - set(filters)
    if dropped:
        print("  (dropped unsupported filters:", dropped, ")")

    return plan.get("search_text", question), filters or None

In [13]:
for q in ["how do I testify at the hearing?",
          "show me the official filed PDF about the human rights commission",
          "are there any cancelled hearings?",
          "what hearings are happening at 4 pm?"]:
    search_text, filters = extract_search_plan(q)
    print(f"\nQ: {q}\n  search_text: {search_text!r}\n  filters: {filters}")


Q: how do I testify at the hearing?
  search_text: 'testify at the hearing in Boston public notices'
  filters: None

Q: show me the official filed PDF about the human rights commission
  search_text: 'official filed PDF about the human rights commission'
  filters: {'source_type': 'pdf'}

Q: are there any cancelled hearings?
  search_text: 'are there any hearings in Boston that are cancelled'
  filters: {'cancelled': True}

Q: what hearings are happening at 4 pm?
  search_text: 'hearings happening at 4 pm'
  filters: None


RAG pipeline: 

In [14]:
def answer(question, k=4):
    """Returns (answer_text, hits). hits are in the same order as the [1..k] citations."""
    # 1. planning
    search_text, filters = extract_search_plan(question)

    # 2. retrieving
    hits = vectorstore.similarity_search(search_text, k=k, filter=filters)

    # 3. filter matching nothing 
    if not hits:
        return (
            "I couldn't find any public notices matching that. "
            f"(searched for {search_text!r} with filters {filters})"
        ), []

    # 4. numbering chunks so model can cite them
    context = "\n\n".join(
        f"[{i + 1}] (source: {d.metadata.get('source_type')}, "
        f"notice {d.metadata.get('notice_id')})\n{d.page_content}"
        for i, d in enumerate(hits)
    )

    # 5. generating 
    prompt = f"""Answer the question using only the context below.
If the context does not contain the answer, say you don't know.
Cite the source number after each individual claim, not once at the end.
If a claim draws on multiple sources, cite all of them.

Context:
{context}

Question: {question}"""

    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content, hits


def print_sources(hits):
    if not hits:
        return
    print("\nSources:")
    for i, h in enumerate(hits, 1):
        m = h.metadata
        title = (m.get("title") or "").replace(" | Boston.gov", "")
        where = m.get("file_label") or "notice webpage"
        print(f" [{i}] [{m.get('source_type')}] {where} - {title}")
        print(f"     {m.get('detail_url')}")

End to end

In [18]:
for q in ["how do I testify at the hearing?", # normal question
          "are there any cancelled hearings?", # filter matching nothing
          "show me the official filed PDF about the human rights commission"]: #asking for a document instead of a fact
    print("\nQ:", q)
    ans, hits = answer(q)
    print(ans)
    if hits:
            print("\nSources:")
            for i, h in enumerate(hits, 1):
                m = h.metadata
                title = (m.get("title") or "").replace(" | Boston.gov", "")
                where = m.get("file_label") or "notice webpage"
                print(f" [{i}] [{m.get('source_type')}] {where}")
                print(f"     {m.get('detail_url')}")


Q: how do I testify at the hearing?
To testify at the hearing, you can attend in person or virtually. If you wish to testify in person, you should arrive five minutes before the call of the hearing to sign up and bring fifteen copies of any written documentation you wish to present. If you prefer to testify virtually via videoconference, you need to email the staff contact for a link and instructions. [1][2]

Sources:
 [1] [pdf] Official Filed Posting
     https://www.boston.gov/public-notices/16600326
 [2] [page_text] notice webpage
     https://www.boston.gov/public-notices/16600326
 [3] [pdf] Official Filed Posting
     https://www.boston.gov/public-notices/16600326
 [4] [page_text] notice webpage
     https://www.boston.gov/public-notices/16600326

Q: are there any cancelled hearings?
I couldn't find any public notices matching that. (searched for 'are there any hearings in Boston that are cancelled?' with filters {'cancelled': True})

Q: show me the official filed PDF about the h